In [445]:
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import RobustScaler
from sklearn.metrics.pairwise import cosine_distances
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt
import numpy as np

# load data

In [446]:
df = pd.read_csv('/Users/connorhall/datasets/inst414/module 3 assignment/Public.csv',
                 usecols=['SPECIES', 'AIRPORT', 'STATE', 'INCIDENT_DATE'])
df = df.dropna()

# remove unknown species
df = df[(~df['SPECIES'].str.contains('unknown|perching birds', case=False)) &
        (~df['AIRPORT'].str.contains('unknown', case=False))]
df

,INCIDENT_DATE,AIRPORT,STATE,SPECIES
8,1992-05-03,NORMAN Y. MINETA SAN JOSE INTL ARPT,CA,American robin
11,1995-04-14,ALEXANDRIA INTL,LA,Blackbirds
13,1994-09-01,SYRACUSE HANCOCK INTL,NY,Gulls
14,1990-09-17,OAKLAND COUNTY INTL,MI,Mourning dove
15,1990-07-13,LOS ANGELES INTL,CA,American kestrel
...,...,...,...,...
339266,2025-10-02,EVANSVILLE REGIONAL,IN,Chimney swift
339267,2025-10-01,SEATTLE-TACOMA INTL,WA,Cedar waxwing
339268,2025-10-02,CHICAGO O'HARE INTL ARPT,IL,Yellow-rumped warbler
339269,2025-10-03,ROANOKE REGNL ARPT/WOODRUM FIELD,VA,European starling


### filter data

In [447]:
# remove low collision count for states and species
#df = df[df.groupby('STATE')['STATE'].transform('count') >= 300]
df = df[df.groupby('SPECIES')['SPECIES'].transform('count') >= 100]

# species must appear in multiple places- avoids exotic species outliers (e.g. introduced species in Hawaii)
df = df[df.groupby('SPECIES')['STATE'].transform('nunique') >= 5]

df = df[['STATE', 'SPECIES']]
df

,STATE,SPECIES
8,CA,American robin
11,LA,Blackbirds
13,NY,Gulls
14,MI,Mourning dove
15,CA,American kestrel
...,...,...
339266,IN,Chimney swift
339267,WA,Cedar waxwing
339268,IL,Yellow-rumped warbler
339269,VA,European starling


# calculate collision counts

In [448]:
counts = pd.crosstab(df['SPECIES'], df['STATE']).stack('STATE')
counts

SPECIES                STATE
American barn owl      AB        0
                       AK        0
                       AL        4
                       AR        7
                       AS        3
                                ..
Yellow-rumped warbler  VT        3
                       WA       15
                       WI        5
                       WV        0
                       WY        0
Length: 10787, dtype: int64

### separate airport and species index

In [449]:
species = counts.index.get_level_values('SPECIES').tolist()
airports = counts.index.get_level_values('STATE').tolist()

counts_df = pd.DataFrame({'Species': species, 'Airport': airports})
counts_df['Count'] = counts.values
counts_df

,Species,Airport,Count
0,American barn owl,AB,0
1,American barn owl,AK,0
2,American barn owl,AL,4
3,American barn owl,AR,7
4,American barn owl,AS,3
...,...,...,...
10782,Yellow-rumped warbler,VT,3
10783,Yellow-rumped warbler,WA,15
10784,Yellow-rumped warbler,WI,5
10785,Yellow-rumped warbler,WV,0


### add collision counts to new dataframe format

In [450]:
# create lists of unique species names and airport names
unique_sp = sorted(list(set(species)))
unique_air = sorted(list(set(airports)))

# create list of collision count lists
counts_list = []
for sp in unique_sp:
    sp_filter = counts_df[counts_df['Species'] == sp]
    counts_list.append(sp_filter['Count'].tolist())
    
# create new dataframe format
collisions_df = pd.DataFrame(counts_list, columns=unique_air, index=unique_sp)
collisions_df

,AB,AK,AL,AR,AS,AZ,BC,CA,CO,CT,DC,DE,FL,FN,GA,GU,HI,IA,ID,IL,IN,KS,KY,LA,MA,MB,MD,ME,MH,MI,MN,MO,MP,MS,MT,NC,ND,NE,NH,NJ,NL,NM,NS,NV,NY,OH,OK,ON,OR,PA,PR,QC,RI,SC,SD,SK,TN,TX,UM,UT,VA,VI,VT,WA,WI,WV,WY
American barn owl,0,0,4,7,3,8,0,783,35,0,14,0,40,4,47,1,399,0,9,8,2,1,6,12,2,0,1,0,0,2,0,11,0,8,0,9,0,1,0,36,0,1,0,5,171,8,5,0,199,3,1,0,0,0,1,0,66,116,0,102,5,0,0,151,0,0,1
American coot,0,0,0,2,0,29,0,95,8,0,0,0,12,0,5,0,0,0,5,23,5,1,8,2,0,0,0,0,0,1,26,10,0,1,1,1,5,5,0,1,0,0,0,22,21,1,1,0,13,3,1,0,0,1,3,0,9,23,0,49,3,0,0,19,3,1,0
American crow,0,2,5,2,0,1,0,96,9,7,8,0,15,2,16,0,0,4,2,29,10,1,23,3,13,0,7,17,0,16,25,6,0,2,3,18,3,6,7,21,0,1,0,0,58,19,3,0,59,26,0,0,2,4,1,0,25,56,0,4,15,0,14,68,9,29,0
American golden-plover,0,18,1,1,0,0,0,1,0,6,7,0,3,0,4,0,0,3,0,41,5,3,3,5,12,0,3,12,0,22,25,10,2,1,1,4,1,3,3,7,0,0,0,0,43,13,0,0,0,12,0,0,3,1,0,0,1,36,0,0,1,3,3,1,18,0,0
American goldfinch,0,0,1,0,0,0,0,9,2,4,1,0,2,0,4,0,0,6,2,6,4,0,7,2,1,0,4,3,0,9,2,1,0,0,1,16,2,1,4,6,0,0,0,0,14,5,1,1,4,7,0,0,1,0,2,0,2,3,0,1,1,0,0,12,4,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Woodchuck,0,0,0,0,0,0,0,0,0,2,1,0,0,0,0,0,0,1,0,13,8,0,4,0,0,0,33,1,0,14,0,0,0,0,0,2,0,1,1,31,0,0,0,0,20,40,0,0,0,31,0,0,0,2,0,0,5,0,0,0,8,0,0,0,0,1,0
Yellow warbler,0,0,0,1,0,1,0,26,3,0,0,0,6,0,0,0,0,0,0,2,1,0,1,0,1,0,0,0,0,4,2,4,0,1,0,5,0,0,0,5,0,0,0,0,11,4,1,0,4,3,0,0,0,0,0,0,6,5,0,2,0,0,0,5,1,0,0
Yellow-bellied sapsucker,0,0,1,0,0,0,0,0,0,2,2,0,4,0,7,0,0,1,0,8,1,0,1,0,3,0,0,2,0,3,4,2,0,0,0,9,1,0,0,16,0,0,0,0,37,1,0,1,0,9,0,0,0,0,0,0,1,1,0,0,2,0,0,0,3,0,0
Yellow-crowned night heron,0,0,1,0,0,0,0,1,0,0,0,0,88,1,2,0,0,0,0,0,0,0,0,5,1,0,0,0,0,0,0,0,0,0,0,2,0,0,0,2,0,0,0,0,12,0,0,0,0,0,3,0,0,0,0,0,1,27,0,0,0,11,0,0,0,0,0


### verify output

In [451]:
# compare sum of collisions from counts_df and collisions_df
counts_1 = counts_df.groupby('Species')['Count'].sum().values.tolist()
counts_2 = collisions_df.sum(axis=1).values.tolist()
counts_1 == counts_2

True

# clustering

In [452]:
scaler = RobustScaler()
scaled_data = scaler.fit_transform(collisions_df)

model = AgglomerativeClustering(n_clusters=5, metric='cosine', linkage='complete')
labels = model.fit_predict(scaled_data)

pd.DataFrame(labels).value_counts()

0
3    68
0    33
1    28
2    16
4    16
Name: count, dtype: int64

### verify that cluster items are mostly similar

In [453]:
copy_df = collisions_df.copy()
copy_df['label'] = labels

In [454]:
for label in range(len(np.unique(labels))):
    row_count = 0
    dissimilar_count = 0

    for sp in copy_df[copy_df['label'] == label].index.tolist():
        row_count += 1
        airport_list1 = counts_df[(counts_df['Species'] == sp) & (counts_df['Count'] > 0)]['Airport'].values.tolist()
        
        for sp2 in copy_df[copy_df['label'] == label].index.tolist():
            if sp2 == sp:
                continue
            
            airport_list2 = counts_df[(counts_df['Species'] == sp2) & (counts_df['Count'] > 0)]['Airport'].values.tolist()
            # if neither species appears in the same state
            if len(set(airport_list1) & set(airport_list2)) == 0:
                dissimilar_count+=1
                
    print(f'cluster cat: {label}, dissimilarities: {dissimilar_count}, total rows: {row_count}')
    

cluster cat: 0, dissimilarities: 4, total rows: 33
cluster cat: 1, dissimilarities: 2, total rows: 28
cluster cat: 2, dissimilarities: 0, total rows: 16
cluster cat: 3, dissimilarities: 2, total rows: 68
cluster cat: 4, dissimilarities: 0, total rows: 16


In [456]:
copy_df[copy_df['label'] == 2]

,AB,AK,AL,AR,AS,AZ,BC,CA,CO,CT,DC,DE,FL,FN,GA,GU,HI,IA,ID,IL,IN,KS,KY,LA,MA,MB,MD,ME,MH,MI,MN,MO,MP,MS,MT,NC,ND,NE,NH,NJ,NL,NM,NS,NV,NY,OH,OK,ON,OR,PA,PR,QC,RI,SC,SD,SK,TN,TX,UM,UT,VA,VI,VT,WA,WI,WV,WY,label
American coot,0,0,0,2,0,29,0,95,8,0,0,0,12,0,5,0,0,0,5,23,5,1,8,2,0,0,0,0,0,1,26,10,0,1,1,1,5,5,0,1,0,0,0,22,21,1,1,0,13,3,1,0,0,1,3,0,9,23,0,49,3,0,0,19,3,1,0,2
American wigeon,0,11,0,0,0,1,2,37,4,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,3,2,0,0,1,0,1,0,0,0,0,1,0,1,2,2,1,0,8,0,0,0,0,0,0,0,0,2,0,18,0,0,0,12,1,0,0,2
Brazilian free-tailed bat,0,0,4,9,0,76,0,191,13,0,0,0,176,2,107,0,0,0,0,1,0,0,0,2,0,0,0,0,0,1,1,0,0,0,0,50,0,0,0,0,0,6,0,29,1,0,1,0,3,1,0,0,0,5,0,0,13,465,0,50,3,0,0,2,0,0,0,2
Brewer's blackbird,0,0,0,0,0,22,0,30,7,0,0,0,0,0,0,0,0,0,8,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,1,0,8,0,0,1,0,9,0,0,0,0,0,3,0,0,5,0,7,0,0,0,0,0,0,0,2
California gull,1,0,0,0,0,0,0,181,3,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,10,1,1,0,0,0,0,0,0,11,0,0,0,0,30,0,0,0,0,0,0,0,0,0,0,97,0,0,0,9,0,0,1,2
Dunlin,0,3,1,0,0,0,2,54,2,0,1,0,26,0,0,0,0,0,0,6,2,0,0,2,13,0,0,0,0,1,0,1,0,0,0,0,0,0,0,2,0,0,0,0,8,2,0,0,8,3,0,0,0,1,0,0,1,1,0,0,0,0,0,9,0,0,0,2
Gadwall,0,1,1,2,0,1,0,17,11,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,3,9,0,1,0,0,3,1,0,2,0,0,0,1,13,0,1,0,8,0,0,0,0,0,0,0,7,8,0,25,0,0,0,3,1,0,0,2
Green-winged teal,0,12,0,1,0,2,0,28,2,1,0,1,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,2,0,0,2,4,0,0,1,0,0,1,0,10,0,0,0,2,10,0,0,0,4,4,0,0,1,1,1,0,1,8,0,23,0,0,1,10,0,0,1,2
New World wood-warblers,0,2,0,2,0,3,0,15,0,0,0,0,12,0,1,0,0,0,3,1,2,1,0,2,0,0,0,2,0,8,2,1,0,0,0,1,0,0,0,4,0,0,0,0,9,15,4,0,3,3,0,0,0,0,0,0,1,10,0,2,0,0,0,4,1,0,0,2
Northern pintail,0,14,0,1,0,5,0,149,4,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,9,0,0,1,0,2,1,0,2,0,0,0,2,0,0,2,0,16,2,0,0,0,0,0,0,2,6,0,26,0,0,0,8,0,0,0,2
